In [23]:
import nest_asyncio
nest_asyncio.apply()

In [24]:
from seleniumbase import Driver
import time
import io
import pandas as pd
from bs4 import BeautifulSoup

URL_NAME = {"standard": "stats"}

def get_fbref_player_table(driver, comp_id, comp_name, season, stat_type):
    url_segment = URL_NAME.get(stat_type, stat_type)
    url = f"https://fbref.com/en/comps/{comp_id}/{season}/{url_segment}/{season}-{comp_name}-Stats"

    driver.get(url)
    time.sleep(5)
    html = driver.page_source

    soup = BeautifulSoup(html, "html.parser")
    comments = soup.find_all(string=lambda text: isinstance(text, str) and "<table" in text)

    table_html = None
    for c in comments:
        if f'id="stats_{stat_type}"' in c:
            table_html = c
            break

    if table_html is None:
        raise ValueError(f"Tabela stats_{stat_type} não encontrada para {season}")

    df = pd.read_html(io.StringIO(str(table_html)))[0]
    df.columns = ['_'.join(col).strip() if 'Unnamed' not in col[0] else col[1]
                  for col in df.columns]
    df = df[df.iloc[:, 0] != "Rk"]
    df["season"] = season
    df["stat_type"] = stat_type
    return df


# --- loop completo, com UMA só sessão de browser ---
seasons = ["2022-2023", "2023-2024", "2024-2025"]
stat_types = ["standard", "shooting", "passing", "passing_types",
              "gca", "defense", "possession", "misc"]

results = {}  # vai guardar um DataFrame por stat_type, já com as 3 épocas juntas

driver = Driver(uc=True, headless=True)
try:
    for stat_type in stat_types:
        dfs_desta_categoria = []
        for season in seasons:
            print(f"A buscar {stat_type} de {season}...")
            try:
                df = get_fbref_player_table(driver, 9, "Premier-League", season, stat_type)
                dfs_desta_categoria.append(df)
            except Exception as e:
                print(f"  ERRO em {stat_type}/{season}: {e}")
            time.sleep(4)  # pausa entre pedidos, mesmo reaproveitando a sessão
        results[stat_type] = pd.concat(dfs_desta_categoria, ignore_index=True)
finally:
    driver.quit()

print("Concluído! Tabelas obtidas:", {k: v.shape for k, v in results.items()})

A buscar standard de 2022-2023...
A buscar standard de 2023-2024...
A buscar standard de 2024-2025...
A buscar shooting de 2022-2023...
A buscar shooting de 2023-2024...
A buscar shooting de 2024-2025...
A buscar passing de 2022-2023...
A buscar passing de 2023-2024...
A buscar passing de 2024-2025...
A buscar passing_types de 2022-2023...
A buscar passing_types de 2023-2024...
A buscar passing_types de 2024-2025...
A buscar gca de 2022-2023...
A buscar gca de 2023-2024...
A buscar gca de 2024-2025...
A buscar defense de 2022-2023...
A buscar defense de 2023-2024...
A buscar defense de 2024-2025...
A buscar possession de 2022-2023...
A buscar possession de 2023-2024...
A buscar possession de 2024-2025...
A buscar misc de 2022-2023...
A buscar misc de 2023-2024...
A buscar misc de 2024-2025...
Concluído! Tabelas obtidas: {'standard': (1723, 27), 'shooting': (1723, 21), 'passing': (1723, 31), 'passing_types': (1723, 26), 'gca': (1723, 27), 'defense': (1723, 27), 'possession': (1723, 31),

In [25]:
import sqlite3
conn = sqlite3.connect("tfc_dados.db")
for stat_type, df in results.items():
    df.to_sql(f"fbref_{stat_type}", conn, if_exists="replace", index=False)
conn.close()
print("Guardado em tfc_dados.db")

Guardado em tfc_dados.db
